# Olist E-Commerce Analysis
## Notebook 2 — Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA = 'C:/Users/loayi/OneDrive/Desktop/Workspace/olist-analysis/data/'

customers    = pd.read_csv(DATA + 'olist_customers_dataset.csv')
orders       = pd.read_csv(DATA + 'olist_orders_dataset.csv')
order_items  = pd.read_csv(DATA + 'olist_order_items_dataset.csv')
payments     = pd.read_csv(DATA + 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(DATA + 'olist_order_reviews_dataset.csv')
products     = pd.read_csv(DATA + 'olist_products_dataset.csv')
sellers      = pd.read_csv(DATA + 'olist_sellers_dataset.csv')
translations = pd.read_csv(DATA + 'product_category_name_translation.csv')

print('loaded')

## Step 1 — Filter to Delivered Orders Only

In [ ]:
orders = orders[orders['order_status'] == 'delivered']
print(orders.shape)

## Step 2 — Parse Date Columns

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes)

## Step 3 — Fix Product Categories

In [ ]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products = pd.merge(products, translations, on='product_category_name', how='left')
products.isnull().sum()

## Step 4 — Build Master DataFrame

In [ ]:
result = pd.merge(orders, order_items, how='left', on='order_id')
result = pd.merge(result, products, how='left', on='product_id')
result = pd.merge(result, customers, how='left', on='customer_id')
result = pd.merge(result, sellers, how='left', on='seller_id')
print(result.shape)

## Step 5 — Verify & Save

In [ ]:
result = result.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

result['shipping_limit_date'] = pd.to_datetime(result['shipping_limit_date'])

print('Nulls:')
print(result.isnull().sum()[result.isnull().sum() > 0])
print()
print('Shape:', result.shape)
print('Unique orders:', result['order_id'].nunique())

result.to_csv(DATA + 'master.csv', index=False)
print('Saved to master.csv')